# Stack-and-Scale: Structural Merging for Asynchronous Pretraining of Decoder-Only LMs

**Thesis artifact** — *Asynchronous Training of Decoder-Only Language Models Through Data Partitioning and Model Merging* (BRAC University, CSE400).
Companion to the **SAP Framework Report**; the experimental driver lives in `SAP_Test.ipynb`.

---

## What this notebook is

This is a **literate, self-contained, runnable reference implementation of the merge algorithm** at the centre of the thesis: **Stack-and-Scale (SAP)**. Where `SAP_Test.ipynb` is the end-to-end experiment (Google Drive, TinyStories, 30-minute GPU shard runs split across Colab accounts), this notebook strips all of that away and keeps only **the algorithm and its proof**, explained line by line.

Everything here runs **on CPU in well under a minute** (only `torch` and `numpy` are required) — clone the repo, "Run All", and watch the exactness gates pass.

## The one-sentence idea

> Both core sublayers of a transformer block are **sums of independent units** — an FFN is a sum over neurons, attention is a sum over heads — so units from *separately trained* models can be laid **side by side** in one wider sublayer, and a single scalar on the output-side weights turns the widened sublayer into the **exact weighted average** of its parents' functions. No gates, no routers, no permutation alignment, no retraining.

## How to read it

The notebook alternates **explanation (markdown)** with **implementation (code)**. Every function from the validated prototype is reproduced verbatim, then dissected — the markdown explains *why each line is what it is* and *which part of the report it realises*. Read top to bottom.

| Section | Contents | Report § |
|---|---|---|
| 0 | The central idea: additivity, the Stack-and-Scale identity, why it needs no gate | §4 |
| 1 | The family skeleton — what must match across models, and why | §5 |
| 2 | The model family in PyTorch — the architecture that *makes merging possible* | §5.1, FAQ |
| 3 | **The merge operator** — `absorb_norm_gains` + `merge_models` (the heart) | §6.4 |
| 4 | The no-cheating machinery — exactness gates to ~1e-16 | §8.2, §14.3 |
| 5 | Runnable demonstrations — implicit gating, width-heterogeneity, closure, growth, NumPy ground truth | §8, §9 |
| 6 | Seed → shard branching (the asynchrony enabler) | §6.2 |
| 7 | The α bookkeeping and continual merging | §7 |
| 8 | End-to-end on synthetic data — the merge on *trained* weights | §6 |
| 9 | Summary, requirement map, honest limits, references | §12, §16 |


## 0 · The central idea (the maths the code makes physical)

The whole framework rests on one structural fact about transformers: **the two sublayers inside a block are sums of independent units that never talk to each other inside the layer.** They only read a shared input and *add* their outputs into a shared residual stream. That additivity is what lets us concatenate units from different models.

### 0.1 The FFN is a sum over neurons

For a SwiGLU FFN with hidden width `d_ff`:

```
FFN(x) = W_down · ( SiLU(W_gate x) ⊙ (W_up x) )
       = Σ_i  [ SiLU(w_gate_i · x) · (w_up_i · x) ] · w_down_i        # sum over the d_ff neurons
```

Hidden unit `i` owns **row `i`** of `W_gate`, **row `i`** of `W_up` (what it listens for) and **column `i`** of `W_down` (the vector it writes into the residual stream when it fires). The neurons do not interact; the layer output is the plain sum of their contributions.

### 0.2 Multi-head attention is a sum over heads

```
MHA(x) = [h_1 | h_2 | … | h_H] · W_O = Σ_h  h_h · W_O^(h)            # sum over the H heads
```

`W_O` is not one entangled matrix — it is `H` **row-blocks** of `d_head` rows, one private "translator" `W_O^(h)` per head that converts that head's output into residual-stream coordinates. Inside each head, the Q/K/V projections, the softmax, and RoPE are entirely self-contained; heads share nothing but the input `x`.

### 0.3 The Stack-and-Scale identity

Give two models reading the same `d_model` stream. Define a merged sublayer by **concatenating** the units and **scaling only the output side** by convex weights α and (1−α):

**FFN** (widths `d_ff^A`, `d_ff^B` → `d_ff^A + d_ff^B`):
```
W_gate^m = [ W_gate^A ; W_gate^B ]          (rows stacked — UNSCALED, input side)
W_up^m   = [ W_up^A   ; W_up^B   ]          (rows stacked — UNSCALED, input side)
W_down^m = [ α·W_down^A | (1−α)·W_down^B ]  (cols stacked — SCALED,   output side)
```
**MHA** (heads `H_A`, `H_B` → `H_A + H_B`):
```
W_Q^m = [ W_Q^A | W_Q^B ]   (and W_K, W_V)  (head columns side by side — UNSCALED, input side)
W_O^m = [ α·W_O^A ; (1−α)·W_O^B ]            (head rows stacked — SCALED,   output side)
```
Then for **every** input `x`:  `merged_sublayer(x) = α·sublayer_A(x) + (1−α)·sublayer_B(x)`, **exactly** (to floating-point rounding).

**Why input-side unscaled, output-side scaled.** Q/K/V (and gate/up) sit on the *input side* of a unit; touching them would change what the unit *computes* internally (its queries, its softmax pattern, its activation). `W_O` (and `W_down`) sit on the *output side*, where contributions combine **linearly** — a scalar there reweights a unit's whole contribution without disturbing its internal computation. Convex weights (Σα = 1) keep the sublayer's output magnitude in the regime every downstream layer was trained for; an unscaled sum would roughly double each contribution and the error would compound geometrically with depth.

### 0.4 Why there is no gate, and why permutation symmetry is sidestepped

- **No gate needed.** Gated/channel-selection merging exists only to decide which units survive when the merged model must stay the *original size*. SAP drops the size constraint, so every unit survives and routing is *implicit and free*: a SiLU neuron whose feature is absent outputs ≈0 and contributes nothing; a head attends with its trained pattern regardless of its neighbours. The trained nonlinearity **is** the gate (demonstrated numerically in §5.1).
- **Permutation symmetry sidestepped.** Naive weight averaging fails because it forces unit `i` of model A to be *summed* with unit `i` of model B, and across independent runs those units learn unrelated features. **Stacking never sums two units** — unit identities are irrelevant, so no Git-Re-Basin-style alignment is required.

### 0.5 The one honest limit

Exactness is **per sublayer, given the same input**. Composed across depth it does *not* make the merged network the average of the two full networks: from layer 2 on, every sublayer reads a *blended* residual stream neither parent saw in training. This **composition gap** — not the operator — is the real scientific risk, treated as the central testable hypothesis (report §12). The seed warm-up (§6) limits it; the optional healing pass (companion notebook) closes it; this notebook keeps the *pure* operator uncontaminated so any measured quality is attributable to the merge alone.


In [1]:
# ---- Section 0 setup: the only dependencies are torch and numpy ----------
import os, math, copy, json
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(0)          # reproducibility for every demo below
np.random.seed(0)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"torch {torch.__version__} | device = {DEVICE}")


torch 2.12.0+cpu | device = cpu


## 1 · The family skeleton

Every model that will ever be merged must belong to one **family**, defined by a fixed **skeleton** chosen once at project start. The skeleton is the contract that makes T2's heads able to read a residual stream written by T1's heads.

### 1.1 Fixed across all members (the skeleton) — and *why* each must match

| Component | Symbol | Why it must be identical |
|---|---|---|
| Number of transformer blocks | `N_LAYERS` (L) | Merging is **blockwise**: block *i* of A merges with block *i* of B. Every block needs a counterpart at the same depth. |
| Residual-stream width | `D_MODEL` | All heads/neurons of all models read this same input vector and write into the same stream; the matrix shapes must compose. |
| Per-head width | `D_HEAD` | Keeps heads uniform so the stacked attention is still a standard fused-attention kernel. |
| Tokenizer / vocabulary | `VOCAB` | Embeddings are **averaged** at merge time, so token id *k* must mean the same string in every model. |
| Positional scheme | RoPE | Per-head and **parameter-free** → merges trivially (nothing to average). |
| Norm type | RMSNorm | Its diagonal gain can be **exactly absorbed** into the next projection (§3.1); LayerNorm's mean-subtraction + bias would make absorption messy. |

### 1.2 Free to differ per model (the *stackable* dimensions)

| Dimension | Symbol | Consequence of merging |
|---|---|---|
| Attention heads per layer | `n_heads` (H) | merged H = Σ Hᵢ |
| FFN hidden width = neuron count | `d_ff` | merged d_ff = Σ d_ffᵢ |
| Data partition, duration, order, hardware, batch size, LR schedule | — | the entire point of **asynchrony** |

Because `n_heads` and `d_ff` are free, **models of different total size merge natively**. The constants below are exactly the family used in `SAP_Test.ipynb`. Treat the skeleton block as immutable — *editing it between sessions is what would break cross-model compatibility.*

### 1.3 Why depth (L) cannot differ

Width units (heads, neurons) combine by **summation**, which is order-free and mergeable. Depth combines by **composition** of functions, which is neither — there is no natural counterpart for block 17 of a 24-block model inside a 12-block model. So depth is part of the skeleton. (Escape hatch, out of scope: grow the shallow model with function-preserving identity blocks, then merge — report §5.3.)


In [2]:
# ============================ FAMILY SKELETON (fixed, report §5.1) ===========
# These six constants are the "family contract". They appear identically in the
# prototype's every Part and must never change between training sessions, because
# they are what let one model's heads read a residual stream written by another's.
VOCAB     = 8192      # tokenizer vocabulary -> embeddings are AVERAGED, so this must match
N_LAYERS  = 8         # L: block i of model A merges with block i of model B
D_MODEL   = 384       # residual-stream width: the shared "bus" every unit reads / writes
D_HEAD    = 64        # per-head width (keeps heads uniform for fused attention)
BLOCK     = 512       # context length (a training detail, not part of the merge contract)
ROPE_BASE = 10000.0   # RoPE base theta; per-head + parameter-free -> merges trivially
NORM_EPS  = 1e-6      # RMSNorm epsilon

# ----------------- STACKABLE DIMS (free per shard, report §5.2) -------------
# A shard picks its width to fit its machine. Merged widths are the SUMS of these.
SHARD_20M = dict(n_heads=6,  d_ff=1536)   # ~22M params  (the narrow family member; T1..T4)
SHARD_40M = dict(n_heads=12, d_ff=3072)   # ~41M params  (the wide   family member; T5)

print("skeleton  : L=%d  d_model=%d  d_head=%d  vocab=%d  (RoPE, RMSNorm, SwiGLU)"
      % (N_LAYERS, D_MODEL, D_HEAD, VOCAB))
print("stackable : narrow %s   wide %s" % (SHARD_20M, SHARD_40M))


skeleton  : L=8  d_model=384  d_head=64  vocab=8192  (RoPE, RMSNorm, SwiGLU)
stackable : narrow {'n_heads': 6, 'd_ff': 1536}   wide {'n_heads': 12, 'd_ff': 3072}


## 2 · The model family in PyTorch

This is an ordinary decoder-only transformer — but it is written so that **every shape that grows under merging is read from a config or a weight tensor, never from a global**. That single discipline is what lets *the same class* instantiate a 22M shard and a 116M thrice-merged model. We build it module by module, and for each one point out the property the merge operator will exploit.


### 2.1 `RMSNorm` — a normaliser whose gain can be *exactly* absorbed

`RMSNorm(x) = g ⊙ x / rms(x)`. The `rms(·)` part is **parameter-free**; the only learned parameter is the **diagonal gain** `g` (one scalar per channel). That diagonal structure is the reason norm gains can be folded into the next linear layer with *zero* approximation before merging (proved in §3.1): for a linear `W`, `W @ (g ⊙ x_hat) = (W · diag(g)) @ x_hat`.

Line notes:
- `self.weight = nn.Parameter(torch.ones(dim))` — `g`, initialised to identity.
- the `x.float()` guard computes the norm in fp32 **only** under fp16/bf16 autocast, while leaving fp64 untouched — important so the double-precision exactness tests in §4 are not silently downcast.
- `torch.rsqrt(mean(x²) + eps)` is `1/rms(x)`; multiplying by `self.weight` applies `g`.


In [3]:
class RMSNorm(nn.Module):
    """RMSNorm: g * x / rms(x). rms() is parameter-free; the diagonal gain g is
    what gets EXACTLY absorbed into the next projections before merging (§6.4)."""
    def __init__(self, dim):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(dim))           # the diagonal gain g
    def forward(self, x):
        # compute in fp32 only when training in half precision; keep fp64 intact
        # so the double-precision exactness tests are not silently downcast
        xf = x.float() if x.dtype in (torch.float16, torch.bfloat16) else x
        out = xf * torch.rsqrt(xf.pow(2).mean(-1, keepdim=True) + NORM_EPS)
        return (self.weight * out.to(x.dtype))


### 2.2 RoPE — rotary positions, per head and parameter-free

Rotary Position Embedding rotates the (even, odd) coordinate pairs of each query/key by an angle that grows with position. Two properties matter for merging:

1. **Parameter-free** — the cos/sin tables are computed from `position × frequency`, not learned, so there is *nothing to merge*. (Contrast learned absolute-position embeddings, which would have to be averaged.)
2. **Per head, identical across the family** — every head of every member uses the same `D_HEAD` and the same `ROPE_BASE`, so after stacking each head keeps its *exact* positional behaviour.

`rope_cos_sin` builds the tables once per forward; `apply_rope` does the pairwise rotation `(x1, x2) → (x1·cos − x2·sin, x1·sin + x2·cos)` and flattens back.


In [4]:
def rope_cos_sin(T, dtype, device):
    """Rotary embedding tables. Identical for every head of every family member
    (same D_HEAD, same base) -> heads keep their exact positional behaviour
    after stacking."""
    inv = 1.0 / (ROPE_BASE ** (torch.arange(0, D_HEAD, 2, device=device).float() / D_HEAD))
    t = torch.arange(T, device=device).float()
    freqs = torch.outer(t, inv)                       # (T, D_HEAD/2)
    return freqs.cos().to(dtype), freqs.sin().to(dtype)

def apply_rope(x, cos, sin):
    # x: (B, H, T, D_HEAD) -> rotate the (even, odd) coordinate pairs
    x1, x2 = x[..., 0::2], x[..., 1::2]
    return torch.stack([x1 * cos - x2 * sin, x1 * sin + x2 * cos], dim=-1).flatten(-2)


### 2.3 `Attention` — and *the head axis*, the key to the attention merge

This is the most important module to understand structurally, because the merge concatenates along its head axis. The subtlety is **PyTorch's storage convention**: `nn.Linear(in, out)` stores its weight as `(out_features, in_features)` — *transposed* relative to maths notation. Trace the shapes:

- `wq, wk, wv = nn.Linear(D_MODEL, H·D_HEAD)` → weight shape `(H·D_HEAD, D_MODEL)`. The output dimension `H·D_HEAD` is the **rows**, split into `H` contiguous **row-blocks of `D_HEAD`** — one block per head. Stacking heads from another model therefore appends rows → `torch.cat(dim=0)`.
- `wo = nn.Linear(H·D_HEAD, D_MODEL)` → weight shape `(D_MODEL, H·D_HEAD)`. Here `H·D_HEAD` is the **input** dimension, i.e. the **columns**, split into `H` **column-blocks** — head *h*'s private translator `W_O^(h)`. Stacking heads appends columns → `torch.cat(dim=1)`.

Inside `forward`, each head computes its own Q, K, V, applies RoPE, and runs a **causal softmax that lives entirely inside the head** — which is exactly why stacking survives the nonlinearity that would destroy attention *averaging*. `self.wo(y)` is precisely `Σ_h head_h · W_O^(h)` (report §4.2).

**`self.h = n_heads` is read from the config**, and the projections are sized from it — the module never consults a global head count, so a merged 36-head block instantiates and runs through the identical class.


In [5]:
class Attention(nn.Module):
    """Multi-head attention. THE HEAD AXIS (report FAQ Q6):
    wq/wk/wv weight has shape (H*D_HEAD, D_MODEL) -> ROW blocks of D_HEAD rows
    per head (because nn.Linear stores (out,in)).  wo weight has shape
    (D_MODEL, H*D_HEAD) -> COLUMN blocks per head.  Merging concatenates along
    exactly these block axes."""
    def __init__(self, n_heads):
        super().__init__()
        self.h = n_heads                                       # read from cfg, never a global
        self.wq = nn.Linear(D_MODEL, n_heads * D_HEAD, bias=False)
        self.wk = nn.Linear(D_MODEL, n_heads * D_HEAD, bias=False)
        self.wv = nn.Linear(D_MODEL, n_heads * D_HEAD, bias=False)
        self.wo = nn.Linear(n_heads * D_HEAD, D_MODEL, bias=False)
    def forward(self, x, cos, sin):
        B, T, _ = x.shape
        q = self.wq(x).view(B, T, self.h, D_HEAD).transpose(1, 2)  # (B,H,T,d)
        k = self.wk(x).view(B, T, self.h, D_HEAD).transpose(1, 2)
        v = self.wv(x).view(B, T, self.h, D_HEAD).transpose(1, 2)
        q, k = apply_rope(q, cos, sin), apply_rope(k, cos, sin)
        # softmax happens INSIDE each head -> untouched by stacking (report §4.3)
        y = F.scaled_dot_product_attention(q, k, v, is_causal=True)
        y = y.transpose(1, 2).reshape(B, T, self.h * D_HEAD)
        return self.wo(y)   # = sum over heads of head_h @ Wo_block_h  (report §4.2)


### 2.4 `FFN` (SwiGLU) — and *the neuron axis*

The same story one dimension over. Hidden unit `i` owns **row `i`** of `w_gate` and **row `i`** of `w_up` (both `nn.Linear(D_MODEL, d_ff)` → weight `(d_ff, D_MODEL)`, so neurons are rows → stack with `cat(dim=0)`), and **column `i`** of `w_down` (`nn.Linear(d_ff, D_MODEL)` → weight `(D_MODEL, d_ff)`, so neurons are columns → stack with `cat(dim=1)`).

`forward` is `w_down( SiLU(w_gate x) ⊙ w_up x )` — the output is the **sum of per-neuron contributions**, so neurons from another model can simply be appended. `d_ff` is read from the config for the same shape-driven reason as the head count.


In [6]:
class FFN(nn.Module):
    """SwiGLU FFN. Hidden unit i owns row i of w_gate, row i of w_up and
    column i of w_down; the output is the SUM of unit contributions (report §4.1),
    which is why units from different models can simply be stacked."""
    def __init__(self, d_ff):
        super().__init__()
        self.w_gate = nn.Linear(D_MODEL, d_ff, bias=False)    # rows = neurons (input side)
        self.w_up   = nn.Linear(D_MODEL, d_ff, bias=False)    # rows = neurons (input side)
        self.w_down = nn.Linear(d_ff, D_MODEL, bias=False)    # cols = neurons (output side)
    def forward(self, x):
        return self.w_down(F.silu(self.w_gate(x)) * self.w_up(x))


### 2.5 `Block` — the residual stream *is* the sum that stacking exploits

A block is the standard pre-norm pattern:

```
x = x + attn(attn_norm(x))     # add the attention sublayer's contribution
x = x + ffn(ffn_norm(x))       # add the FFN sublayer's contribution
```

Note the `+`: every sublayer **adds** into the residual stream. That residual addition is the very operation Stack-and-Scale exploits — concatenating units widens the sum, and α-scaling reweights the addends, while the stream's shape (`D_MODEL`) is unchanged.


In [7]:
class Block(nn.Module):
    def __init__(self, n_heads, d_ff):
        super().__init__()
        self.attn_norm, self.attn = RMSNorm(D_MODEL), Attention(n_heads)
        self.ffn_norm,  self.ffn  = RMSNorm(D_MODEL), FFN(d_ff)
    def forward(self, x, cos, sin):
        x = x + self.attn(self.attn_norm(x), cos, sin)   # residual stream: a SUM —
        x = x + self.ffn(self.ffn_norm(x))               # the same op stacking exploits
        return x


### 2.6 `GPT` — one family member, sized entirely from its config

The full model: token embedding → `N_LAYERS` blocks → final RMSNorm → tied LM head. The design choices that matter for merging:

- **`self.cfg = dict(cfg)`** records only the *stackable* dims (`n_heads`, `d_ff`); the skeleton lives in the globals. Every block is built from `cfg["n_heads"]` / `cfg["d_ff"]`, so **a merged config with summed widths instantiates the same class**.
- **`self.lm_head.weight = self.embed.weight`** — tied embeddings. There is one embedding matrix (the skeleton fixes `VOCAB × D_MODEL`); it is the single tensor the merge has to *average* rather than stack.
- **GPT-2-style init** scales the *output-side* projections (`wo`, `w_down`) by `1/sqrt(2·N_LAYERS)` so the residual stream does not blow up with depth at initialisation — orthogonal to merging, but it is what the prototype trained with, kept here for fidelity.
- **`n_params()`** lets the growth-law demos in §5 read off parameter counts directly.


In [8]:
class GPT(nn.Module):
    """One member of the model family. cfg = {'n_heads': H, 'd_ff': F}.
    H and d_ff are read from cfg (and implicitly from weight shapes), never
    from globals -> merged (wider) models load and run with the same class."""
    def __init__(self, cfg):
        super().__init__()
        self.cfg = dict(cfg)
        self.embed = nn.Embedding(VOCAB, D_MODEL)
        self.blocks = nn.ModuleList([Block(cfg["n_heads"], cfg["d_ff"]) for _ in range(N_LAYERS)])
        self.final_norm = RMSNorm(D_MODEL)
        self.lm_head = nn.Linear(D_MODEL, VOCAB, bias=False)
        self.lm_head.weight = self.embed.weight          # tied embeddings (skeleton)
        self.apply(self._init)
        # GPT-2-style residual scaling on output-side projections
        for blk in self.blocks:
            for w in (blk.attn.wo.weight, blk.ffn.w_down.weight):
                nn.init.normal_(w, mean=0.0, std=0.02 / math.sqrt(2 * N_LAYERS))
    def _init(self, m):
        if isinstance(m, nn.Linear):
            nn.init.normal_(m.weight, mean=0.0, std=0.02)
        elif isinstance(m, nn.Embedding):
            nn.init.normal_(m.weight, mean=0.0, std=0.02)
    def forward(self, idx, targets=None):
        B, T = idx.shape
        x = self.embed(idx)
        cos, sin = rope_cos_sin(T, x.dtype, x.device)
        for blk in self.blocks:
            x = blk(x, cos, sin)
        x = self.final_norm(x)
        logits = self.lm_head(x)
        if targets is None:
            return logits
        loss = F.cross_entropy(logits.view(-1, VOCAB), targets.view(-1))
        return logits, loss
    def n_params(self):
        return sum(p.numel() for p in self.parameters())


In [9]:
# ---- sanity: the SAME class builds a narrow shard and the wide member -------
_narrow = GPT(SHARD_20M)
_wide   = GPT(SHARD_40M)
print("narrow shard  cfg=%s  -> %5.1fM params" % (_narrow.cfg, _narrow.n_params()/1e6))
print("wide member   cfg=%s -> %5.1fM params" % (_wide.cfg,   _wide.n_params()/1e6))
print("block-0 wq weight shapes:  narrow", tuple(_narrow.blocks[0].attn.wq.weight.shape),
      " wide", tuple(_wide.blocks[0].attn.wq.weight.shape))
del _narrow, _wide


narrow shard  cfg={'n_heads': 6, 'd_ff': 1536}  ->  22.0M params
wide member   cfg={'n_heads': 12, 'd_ff': 3072} ->  40.9M params
block-0 wq weight shapes:  narrow (384, 384)  wide (768, 384)


## 3 · The merge operator

This is the heart of the thesis. The merge is **pure tensor surgery** — no data, no gradients, no training, no gates, no routers. It is the four `torch.cat` calls of §3.2, plus one weighted average for the embedding. The report breaks it into four steps:

1. **Norm-gain absorption** (`absorb_norm_gains`, §3.1) — fold each model's RMSNorm gains into its own projections so every unit sees exactly the normalised input it trained on.
2. **Attention merge** — Q/K/V column-concat (unscaled), `W_O` row-concat (scaled by αᵢ).
3. **FFN merge** — gate/up row-concat (unscaled), `W_down` column-concat (scaled by αᵢ).
4. **Embedding / final-norm** — weighted average (the *single* parameter-level component).

The output is a brand-new family member with `n_heads = Σ Hᵢ` and `d_ff = Σ d_ffᵢ`.


### 3.1 `absorb_norm_gains` — making the norm exact (Step 1)

RMSNorm applies `g ⊙ x_hat` where `x_hat = x / rms(x)`. A projection that reads the normalised vector computes `W @ (g ⊙ x_hat)`. Because `g` is **diagonal**, this is identically equal to `(W · diag(g)) @ x_hat` — so we can *fold* `g` into `W` and set the gain to 1, with **zero approximation**:

```
W @ (g ⊙ x_hat)  ==  (W * g) @ x_hat            # g broadcast over the input (column) dim of W
```

For each block we fold the **attention-norm** gain into `wq, wk, wv`, and the **ffn-norm** gain into `w_gate, w_up` (the projections that read each norm), then reset both gains to 1.

- `lin.weight.data.mul_(g)` — `weight` is `(out, in)`; multiplying by `g` of shape `(in,)` broadcasts over rows, scaling each **input** dimension. Exactly `W · diag(g)`.
- Why this matters for merging: after absorption, every head/neuron of every parent sees precisely the input it was trained on, so when we later set the *merged* block's gain to 1 there is nothing left to approximate. (The naive alternative — averaging raw norm gains across models — is the small error this step removes.)

`@torch.no_grad()` because we are editing weights in place, not building a graph. **This function mutates the model**, so `merge_models` always calls it on *deep copies* (§3.2) — your saved parents are never touched.


In [10]:
@torch.no_grad()
def absorb_norm_gains(model):
    """Step 1 (report §6.4) — fold each block's RMSNorm gain g into the projections
    that read it (W_Q,W_K,W_V get the attention-norm gain; w_gate,w_up get the
    FFN-norm gain), then set the gain to 1.  EXACT because the gain is diagonal
    and rms() is parameter-free: W @ (g*x_hat) == (W*g) @ x_hat.
    After this, every head/neuron sees exactly the normalised input it was
    trained on, even inside the merged model."""
    for blk in model.blocks:
        g = blk.attn_norm.weight.data                    # (D_MODEL,)
        for lin in (blk.attn.wq, blk.attn.wk, blk.attn.wv):
            lin.weight.data.mul_(g)                      # (out,in)*(in,) scales input dims
        blk.attn_norm.weight.data.fill_(1.0)
        g = blk.ffn_norm.weight.data
        for lin in (blk.ffn.w_gate, blk.ffn.w_up):
            lin.weight.data.mul_(g)
        blk.ffn_norm.weight.data.fill_(1.0)
    return model


### 3.2 `merge_models` — Stack-and-Scale, N-way (Steps 2–4)

The whole theory, made physical. Read it next to the identity in §0.3. Walking the function:

- **`assert abs(sum(alphas) - 1.0) < 1e-8`** — convexity (Σα = 1), the constraint that keeps the merged output magnitude in the trained regime (report §7.1).
- **`srcs_models = [absorb_norm_gains(copy.deepcopy(m).cpu()) ...]`** — deep-copy every parent (and move to CPU for memory-safe surgery), then absorb its norms. The originals are never mutated.
- **`cfg = dict(n_heads=Σ Hᵢ, d_ff=Σ d_ffᵢ)`** then **`merged = GPT(cfg)`** — the merged member is just a wider family member; the growth law `n_heads = Σ Hᵢ`, `d_ff = Σ d_ffᵢ` is literally this line.
- **The four `torch.cat` calls** are the entire operator. Mind the axis flip from §2.3/§2.4 (`nn.Linear` is `(out, in)`):
  - `wq/wk/wv` → `cat(dim=0)` **unscaled** — append each model's head **rows** (output dim grows). Input side, so no scaling.
  - `wo` → `cat(dim=1)` **scaled by αᵢ** — append each model's translator **columns** (input dim grows), each block weighted by its α. Output side.
  - `w_gate/w_up` → `cat(dim=0)` **unscaled** — append each model's neuron **rows**.
  - `w_down` → `cat(dim=1)` **scaled by αᵢ** — append each model's neuron **columns**, α-weighted.
- **`tb.attn_norm.weight.fill_(1.0); tb.ffn_norm.weight.fill_(1.0)`** — the merged block's gains are identity because Step 1 already folded every parent's gain into its own (now stacked) projections.
- **`merged.embed.weight.copy_(Σ αᵢ · embedⁱ)`** and the same for `final_norm` — Step 4, the lone parameter-level average. Because embeddings are *tied*, this one tensor also serves as the LM head. (This is the framework's single honest approximation; the shared seed keeps shard embeddings close so the average is mild.)

α appears in **exactly two places** — `wo` and `w_down` — the output side of each sublayer. Everything else is verbatim concatenation. N-way is not a special case: pass `N` models and `N` alphas and the same `torch.cat` lists handle it.


In [11]:
@torch.no_grad()
def merge_models(models, alphas):
    """Stack-and-Scale N-way merge (report §6.4). Pure tensor surgery: no data,
    no gradients, no training, no gates, no routers.
      attention : Q/K/V row-concat (UNSCALED)  | W_O  col-concat (SCALED by alpha_i)
      ffn       : gate/up row-concat (UNSCALED) | W_down col-concat (SCALED by alpha_i)
      embeddings/final norm : weighted average (the only parameter-level step)
    Returns a NEW family member with n_heads = sum(H_i), d_ff = sum(F_i).
    NOTE on concat axes: nn.Linear stores weight as (out_features,in_features),
    so 'stack head/neuron units' = cat dim=0 on input-side matrices and
    cat dim=1 on output-side matrices -- transposed relative to math notation."""
    assert abs(sum(alphas) - 1.0) < 1e-8, "alphas must sum to 1 (convexity, report §7.1)"
    dtype = next(models[0].parameters()).dtype
    # absorb norms on deep copies -- never mutate the parents
    srcs_models = [absorb_norm_gains(copy.deepcopy(m).cpu()) for m in models]
    cfg = dict(n_heads=sum(m.cfg["n_heads"] for m in models),
               d_ff=sum(m.cfg["d_ff"] for m in models))
    merged = GPT(cfg).to(dtype)
    for li in range(N_LAYERS):
        tb, srcs = merged.blocks[li], [m.blocks[li] for m in srcs_models]
        # ---- attention (Step 2): each head keeps its own Q/K/V columns and its
        # own W_O translator slice; only the translator is alpha-scaled
        tb.attn.wq.weight.copy_(torch.cat([s.attn.wq.weight for s in srcs], dim=0))
        tb.attn.wk.weight.copy_(torch.cat([s.attn.wk.weight for s in srcs], dim=0))
        tb.attn.wv.weight.copy_(torch.cat([s.attn.wv.weight for s in srcs], dim=0))
        tb.attn.wo.weight.copy_(torch.cat(
            [a * s.attn.wo.weight for a, s in zip(alphas, srcs)], dim=1))
        # ---- ffn (Step 3): every neuron keeps its trained input rows; its
        # output column is alpha-scaled
        tb.ffn.w_gate.weight.copy_(torch.cat([s.ffn.w_gate.weight for s in srcs], dim=0))
        tb.ffn.w_up.weight.copy_(torch.cat([s.ffn.w_up.weight for s in srcs], dim=0))
        tb.ffn.w_down.weight.copy_(torch.cat(
            [a * s.ffn.w_down.weight for a, s in zip(alphas, srcs)], dim=1))
        # ---- norms were absorbed -> identity gains in the merged block
        tb.attn_norm.weight.fill_(1.0); tb.ffn_norm.weight.fill_(1.0)
    # ---- embeddings + final norm (Step 4): the single averaged component
    merged.embed.weight.copy_(sum(a * m.embed.weight for a, m in zip(alphas, srcs_models)))
    merged.final_norm.weight.copy_(sum(a * m.final_norm.weight
                                       for a, m in zip(alphas, srcs_models)))
    return merged


### 3.3 The merge, line by line → idea map

| Code | Tensor effect | Idea it realises |
|---|---|---|
| `absorb_norm_gains(deepcopy(m))` | fold `g` into `W`, set gain=1, on a copy | Step 1; exact RMSNorm handling without mutating parents |
| `cfg = dict(n_heads=Σ Hᵢ, d_ff=Σ d_ffᵢ)` | merged widths = sums | the growth law (report §9.1) |
| `cat([wq...], dim=0)` (k, v too) | append head **rows**, unscaled | heads keep their trained queries/keys/values (input side) |
| `cat([αᵢ·wo...], dim=1)` | append translator **cols**, α-scaled | per-head `W_O^(h)` reweighted → α-average of contributions (output side) |
| `cat([w_gate/w_up...], dim=0)` | append neuron **rows**, unscaled | neurons keep what they listen for (input side) |
| `cat([αᵢ·w_down...], dim=1)` | append neuron **cols**, α-scaled | per-neuron output vector reweighted (output side) |
| `attn_norm/ffn_norm.fill_(1.0)` | merged gains = identity | gains already absorbed into the stacked projections |
| `embed = Σ αᵢ·embedⁱ` | weighted average | Step 4, the one parameter-level component |

The merge is `O(parameters)` time and allocates one new model; there is no optimisation loop anywhere.


## 4 · The no-cheating machinery

A merge that "works" because of a bug would invalidate the whole thesis. Three independent guards keep the result honest; two of them are exactness *proofs* and live here.

1. **`exactness_test()`** — runs on randomised **double-precision** models *before* any real merge: every merged sublayer must equal the α-weighted average of its parents to ~1e-16, and `merge(merge(A,B),C) == merge(A,B,C)` (closure). If this fails, no downstream number is trustworthy.
2. **`real_merge_block_check()`** — re-verifies the *same* identity on the **actual trained weights** after every real merge, block by block, memory-light.
3. **The weight-averaging baseline** (in the companion experiment notebook) — if structural merging only "worked" trivially, naive parameter averaging would work too; it does not.


### 4.1 `exactness_test` — per-sublayer exactness + closure, to 1e-16

Build two double-precision members of *different width* (`A`: 2 heads / 8 d_ff, `B`: 3 heads / 12 d_ff). Crucially, the RMSNorm gains are **randomised** (`uniform_(0.5, 1.5)`) so Step 1's absorption is genuinely exercised — if absorption were wrong, the error would explode.

For every block we compare the **merged sublayer run through its identity norm** against the **α-weighted sum of the parents run through their own (randomised) norms**:
```
target = α·A.attn(A.attn_norm(x)) + (1−α)·B.attn(B.attn_norm(x))
got    = M.attn(M.attn_norm(x))
worst  = max(worst, |got − target|.max())
```
Then the **closure** check: a 3-way uniform merge must equal the sequential `merge(merge(A,B;½,½), C;⅔,⅓)` *weight for weight* — the property that makes continual merging well-defined (report §7.3). Both must be `< 1e-10` (they land near 1e-15). The function `assert`s, so a broken merge aborts the notebook.


In [12]:
@torch.no_grad()
def exactness_test(verbose=True):
    """Report §14.3 -- the merged sublayer must equal the alpha-weighted average of
    the parent sublayers ON THE SAME INPUT, to double-precision rounding error.
    Norm gains are randomised so the absorption step is genuinely exercised.
    If this fails, the merge is broken and NO downstream result counts."""
    torch.manual_seed(0)
    A = GPT(dict(n_heads=2, d_ff=8)).double()
    B = GPT(dict(n_heads=3, d_ff=12)).double()
    for m in (A, B):
        for blk in m.blocks:
            blk.attn_norm.weight.data.uniform_(0.5, 1.5)   # randomise gains -> test absorption
            blk.ffn_norm.weight.data.uniform_(0.5, 1.5)
    a = 0.35
    M = merge_models([A, B], [a, 1 - a])
    x = torch.randn(2, 7, D_MODEL, dtype=torch.float64)
    cos, sin = rope_cos_sin(7, torch.float64, x.device)
    worst = 0.0
    for li in range(N_LAYERS):
        bA, bB, bM = A.blocks[li], B.blocks[li], M.blocks[li]
        # full attention path incl. each parent's OWN norm vs merged identity-norm path
        tgt = a * bA.attn(bA.attn_norm(x), cos, sin) + (1-a) * bB.attn(bB.attn_norm(x), cos, sin)
        got = bM.attn(bM.attn_norm(x), cos, sin)
        worst = max(worst, (got - tgt).abs().max().item())
        tgt = a * bA.ffn(bA.ffn_norm(x)) + (1-a) * bB.ffn(bB.ffn_norm(x))
        got = bM.ffn(bM.ffn_norm(x))
        worst = max(worst, (got - tgt).abs().max().item())
    # N-way == sequential (closure, report §7.3)
    C = GPT(dict(n_heads=1, d_ff=4)).double()
    M3   = merge_models([A, B, C], [1/3, 1/3, 1/3])
    Mab  = merge_models([A, B], [0.5, 0.5])
    Mseq = merge_models([Mab, C], [2/3, 1/3])
    seq_err = max((p - q).abs().max().item()
                  for p, q in zip(M3.state_dict().values(), Mseq.state_dict().values()))
    ok = worst < 1e-10 and seq_err < 1e-10
    if verbose:
        print(f"[exactness] worst sublayer error      = {worst:.3e}  (must be < 1e-10)")
        print(f"[exactness] sequential-vs-3way error  = {seq_err:.3e}  (must be < 1e-10)")
        print("[exactness] PASS" if ok else "[exactness] *** FAIL -- DO NOT TRUST ANY MERGE ***")
    assert ok, "Exactness test failed -- the merge implementation is wrong."
    return worst, seq_err


### 4.2 `real_merge_block_check` — the same proof on *trained* weights

`exactness_test` proves the *operator* is correct on toy models. After a real merge we also want to prove the identity holds on the *specific trained tensors* we just combined. This function does that block by block, and is deliberately **memory-light**: it casts **one block at a time** to float64 on CPU (never whole multi-hundred-MB models), so it runs on any machine.

Because the trained weights are stored in float32, the residual here is larger than in the fp64 test — expect ~1e-9…1e-6 (float32 storage rounding). A *wrong* merge shows errors around 1e0, so the `assert worst < 1e-5` cleanly separates "correct" from "bug".


In [13]:
@torch.no_grad()
def real_merge_block_check(parents, alphas, merged, n_tokens=6):
    """Same exactness assertion, but on the REAL trained models being merged.
    Run after every real merge. Memory-light: casts ONE block at a time to
    float64 on CPU (never whole models), so it works on any runtime."""
    x = torch.randn(1, n_tokens, D_MODEL, dtype=torch.float64)
    cos, sin = rope_cos_sin(n_tokens, torch.float64, x.device)
    worst = 0.0
    for li in range(N_LAYERS):
        pbs = [copy.deepcopy(p.blocks[li]).cpu().double() for p in parents]
        mb  = copy.deepcopy(merged.blocks[li]).cpu().double()
        tgt = sum(a * P.attn(P.attn_norm(x), cos, sin) for a, P in zip(alphas, pbs))
        got = mb.attn(mb.attn_norm(x), cos, sin)
        worst = max(worst, (got - tgt).abs().max().item())
        tgt = sum(a * P.ffn(P.ffn_norm(x)) for a, P in zip(alphas, pbs))
        got = mb.ffn(mb.ffn_norm(x))
        worst = max(worst, (got - tgt).abs().max().item())
        del pbs, mb
    print(f"[real-merge check] worst sublayer error on trained weights = {worst:.3e}\n"
          f"  (expected ~1e-9..1e-6 = float32 weight-storage rounding;\n"
          f"   a WRONG merge shows errors around 1e0)")
    assert worst < 1e-5, "Real merge failed exactness -- aborting."
    return worst


### 4.3 Run the gate

Run the exactness proof now, before any demonstration. Every figure later in the notebook is only meaningful because this passes.

In [14]:
worst, seq_err = exactness_test()


[exactness] worst sublayer error      = 1.041e-16  (must be < 1e-10)
[exactness] sequential-vs-3way error  = 1.388e-17  (must be < 1e-10)
[exactness] PASS


## 5 · Demonstrations

With the operator proved correct, here are five small, fully-runnable demonstrations that turn each abstract claim into a number you can see.

### 5.1 Implicit gating — the hand-worked FFN example (report §8.1)

The cleanest illustration of *why no gate is needed*. Two tiny 2-neuron ReLU FFNs over a `d_model = 2` stream, merged with α = ½, on the input `x = (1, −2)`:

- **Model A** recognises this input — one neuron fires, output `(1, 3)`.
- **Model B** does **not** — both its neurons land at ≤ 0 and ReLU zeroes them, output `(0, 0)`.

The α-average target is `(0.5, 1.5)`. The merged 4-neuron FFN reproduces it **exactly** — and notice *how*: B's stacked neurons contributed literally nothing, so the merged answer was automatically A-dominated. The trained nonlinearity did the routing; there is no gate network anywhere. (This is plain NumPy, deliberately independent of the `GPT` class.)


In [15]:
import numpy as np
relu = lambda z: np.maximum(z, 0.0)
def tiny_ffn(x, W1, W2):              # W1:(d_ff,d_model) rows=neurons ; W2:(d_model,d_ff) cols=neurons
    return W2 @ relu(W1 @ x)
x   = np.array([1.0, -2.0])
W1A = np.array([[1., 0.], [0., 1.]]);  W2A = np.array([[1., 2.], [3., 4.]])
W1B = np.array([[2., 1.], [-1., 1.]]); W2B = np.array([[1., 0.], [0., 1.]])
a = 0.5
outA, outB = tiny_ffn(x, W1A, W2A), tiny_ffn(x, W1B, W2B)
target = a * outA + (1 - a) * outB
W1m = np.concatenate([W1A, W1B], axis=0)                 # 4 neurons stacked (rows, unscaled)
W2m = np.concatenate([a * W2A, (1 - a) * W2B], axis=1)   # output cols, alpha-scaled
merged = tiny_ffn(x, W1m, W2m)
print("A: preact", W1A @ x, "-> relu", relu(W1A @ x), "-> out", outA)
print("B: preact", W1B @ x, "-> relu", relu(W1B @ x), "-> out", outB, " (B recognises nothing -> fires 0)")
print("target  alpha-average :", target)
print("merged  4-neuron FFN  :", merged)
print("max |merged - target| :", np.abs(merged - target).max(), " (exact)")


A: preact [ 1. -2.] -> relu [1. 0.] -> out [1. 3.]
B: preact [ 0. -3.] -> relu [0. 0.] -> out [0. 0.]  (B recognises nothing -> fires 0)
target  alpha-average : [0.5 1.5]
merged  4-neuron FFN  : [0.5 1.5]
max |merged - target| : 0.0  (exact)


### 5.2 Different-size merging + the growth law

Merge two *real* family members of different width — a 6-head / 1536-d_ff member with a 3-head / 768-d_ff member — into a 9-head / 2304-d_ff model. The operator does not even notice the asymmetry. We confirm two things: the **growth law** holds to the parameter, and `real_merge_block_check` confirms per-sublayer exactness on these (randomly-initialised) weights.

The exact law is `S_merged = E + NORM + Σ Pᵢ`, where `E = VOCAB×D_MODEL` is the shared embedding/LM-head, `NORM = (2·L+1)·D_MODEL` are the RMSNorm gains (**not** stacked — absorbed to identity, so one set survives), and `Pᵢ` is member *i*'s attention+FFN parameters. The report writes this compactly as `E + Σ(Sᵢ−E)`, folding the few-thousand norm params into `Pᵢ`; at billion-parameter scale the difference vanishes, but here we check it to the parameter.

In [16]:
torch.manual_seed(1)
A = GPT(dict(n_heads=6, d_ff=1536))
B = GPT(dict(n_heads=3, d_ff=768))
alpha = 0.4
M = merge_models([A, B], [alpha, 1 - alpha])
print("A %s  +  B %s   ->   merged %s" % (A.cfg, B.cfg, M.cfg))
print("params:  A %.1fM   B %.1fM   merged %.1fM" % (A.n_params()/1e6, B.n_params()/1e6, M.n_params()/1e6))
# Exact growth law: only attention+FFN STACK. E (embedding) is shared, and the
# RMSNorm gains are NOT stacked (absorbed to identity -> one set survives).
E    = VOCAB * D_MODEL                                 # shared embedding / LM-head
NORM = (2 * N_LAYERS + 1) * D_MODEL                    # 2 gains/block + final norm (shared)
stack = lambda m: m.n_params() - E - NORM              # member's truly stackable params
rhs = E + NORM + stack(A) + stack(B)
print("growth law:  E + NORM + (P_A + P_B) = %d   merged = %d   match = %s"
      % (rhs, M.n_params(), rhs == M.n_params()))
real_merge_block_check([A, B], [alpha, 1 - alpha], M)
del A, B, M


A {'n_heads': 6, 'd_ff': 1536}  +  B {'n_heads': 3, 'd_ff': 768}   ->   merged {'n_heads': 9, 'd_ff': 2304}
params:  A 22.0M   B 12.6M   merged 31.5M
growth law:  E + NORM + (P_A + P_B) = 31463808   merged = 31463808   match = True
[real-merge check] worst sublayer error on trained weights = 3.169e-09
  (expected ~1e-9..1e-6 = float32 weight-storage rounding;
   a WRONG merge shows errors around 1e0)


### 5.3 Closure — continual merging equals simultaneous merging

A merged model is itself a standard family member, so it can be merged again. With consistent weighting, `merge(merge(A,B), C)` produces the **same weights** as the one-shot `merge(A,B,C)` — the property that makes "incorporate a new partition months later" well-defined (report §7.3). Here it is, tensor for tensor.

In [17]:
torch.manual_seed(2)
A = GPT(dict(n_heads=2, d_ff=64)); B = GPT(dict(n_heads=3, d_ff=96)); C = GPT(dict(n_heads=1, d_ff=32))
M3   = merge_models([A, B, C], [1/3, 1/3, 1/3])           # one-shot 3-way
Mab  = merge_models([A, B], [0.5, 0.5])                   # sequential ...
Mseq = merge_models([Mab, C], [2/3, 1/3])                 # ... fold in C, weighted 2/3 vs 1/3
diff = max((p - q).abs().max().item()
           for p, q in zip(M3.state_dict().values(), Mseq.state_dict().values()))
print("max |W(3-way) - W(sequential)| = %.2e   closure holds = %s" % (diff, diff < 1e-5))
del A, B, C, M3, Mab, Mseq


max |W(3-way) - W(sequential)| = 7.45e-09   closure holds = True


### 5.4 Parameter growth — the build plan

Attention + FFN parameters **add**; the embedding/LM-head budget `E = VOCAB × D_MODEL` is **shared**, and the RMSNorm gains (`NORM`) are shared too (not stacked). So merged size is exactly `E + NORM + Σ Pᵢ` (the report's `E + Σ(Sᵢ−E)` with the tiny norm term made explicit). The `family_size` formula below is checked against the actual module, then used to print the growth table and the experiment's real lineage (T_m1 ≈ 60M, T_m2 ≈ 79M, T_m3 ≈ 116M — matching the report's build plan, §9.2).

In [18]:
E    = VOCAB * D_MODEL
NORM = (2 * N_LAYERS + 1) * D_MODEL                        # RMSNorm gains: shared, not stacked
def family_size(n_heads, d_ff):
    # per block: attention 4*d_model*d_head*H  + SwiGLU 3*d_model*d_ff  + 2 RMSNorm gains
    per_layer = 4 * D_MODEL * D_HEAD * n_heads + 3 * D_MODEL * d_ff + 2 * D_MODEL
    return E + N_LAYERS * per_layer + D_MODEL              # + final norm; embedding tied -> counted once
assert family_size(**SHARD_20M) == GPT(SHARD_20M).n_params(), "growth formula must match the real module"
P_shard = family_size(**SHARD_20M) - E - NORM             # stackable params of one narrow shard

print("E (shared embedding/LM-head)        = %5.1fM" % (E / 1e6))
print("NORM gains (shared, not stacked)    = %5.3fM" % (NORM / 1e6))
print("one narrow shard  (6 heads, 1536)   = %5.1fM\n" % (family_size(**SHARD_20M) / 1e6))
print(" N narrow shards merged   size (exact)")
for N in range(1, 8):
    print("   %d                    %6.1fM" % (N, (E + NORM + N * P_shard) / 1e6))
print("\n actual experiment lineage:")
for name, (h, f) in [("Tm1 = T1+T2+T3", (18, 4608)), ("Tm2 = Tm1+T4", (24, 6144)), ("Tm3 = Tm2+T5", (36, 9216))]:
    print("  %-16s (n_heads=%2d, d_ff=%4d) -> %6.1fM" % (name, h, f, family_size(h, f) / 1e6))


E (shared embedding/LM-head)        =   3.1M
NORM gains (shared, not stacked)    = 0.007M
one narrow shard  (6 heads, 1536)   =  22.0M

 N narrow shards merged   size (exact)
   1                      22.0M
   2                      40.9M
   3                      59.8M
   4                      78.6M
   5                      97.5M
   6                     116.4M
   7                     135.3M

 actual experiment lineage:
  Tm1 = T1+T2+T3   (n_heads=18, d_ff=4608) ->   59.8M
  Tm2 = Tm1+T4     (n_heads=24, d_ff=6144) ->   78.6M
  Tm3 = Tm2+T5     (n_heads=36, d_ff=9216) ->  116.4M


### 5.5 Independent NumPy ground truth (report Appendix A)

The strongest "no hidden bug" evidence: re-derive the identity **from scratch in NumPy**, never touching the `GPT` class — a completely separate implementation of attention and SwiGLU. Both errors land at double-precision rounding (~1e-15), which means the identities are *exact*, not approximate.

In [19]:
rng = np.random.RandomState(0)
dm, dh = 8, 4
HA, HB = 2, 3
T, alpha = 5, 0.4
xx = rng.randn(T, dm)
def softmax(z):
    z = z - z.max(-1, keepdims=True); e = np.exp(z); return e / e.sum(-1, keepdims=True)
def mha(x, Wq, Wk, Wv, Wo, H):
    Q, K, V = x @ Wq, x @ Wk, x @ Wv
    mask = np.triu(np.ones((T, T)), 1) * -1e9
    heads = []
    for h in range(H):
        sl = slice(h * dh, (h + 1) * dh)
        att = softmax(Q[:, sl] @ K[:, sl].T / np.sqrt(dh) + mask)
        heads.append(att @ V[:, sl])
    return np.concatenate(heads, 1) @ Wo
WqA, WkA, WvA = (rng.randn(dm, HA * dh) for _ in range(3)); WoA = rng.randn(HA * dh, dm)
WqB, WkB, WvB = (rng.randn(dm, HB * dh) for _ in range(3)); WoB = rng.randn(HB * dh, dm)
tgt = alpha * mha(xx, WqA, WkA, WvA, WoA, HA) + (1 - alpha) * mha(xx, WqB, WkB, WvB, WoB, HB)
Wq = np.concatenate([WqA, WqB], 1); Wk = np.concatenate([WkA, WkB], 1); Wv = np.concatenate([WvA, WvB], 1)
Wo = np.concatenate([alpha * WoA, (1 - alpha) * WoB], 0)
print("attention merge max error:", np.abs(mha(xx, Wq, Wk, Wv, Wo, HA + HB) - tgt).max())

dfa, dfb = 6, 10
silu = lambda z: z / (1 + np.exp(-z))
ffn = lambda x, Wg, Wu, Wd: (silu(x @ Wg.T) * (x @ Wu.T)) @ Wd.T
WgA, WuA = rng.randn(dfa, dm), rng.randn(dfa, dm); WdA = rng.randn(dm, dfa)
WgB, WuB = rng.randn(dfb, dm), rng.randn(dfb, dm); WdB = rng.randn(dm, dfb)
tgt = alpha * ffn(xx, WgA, WuA, WdA) + (1 - alpha) * ffn(xx, WgB, WuB, WdB)
Wg = np.concatenate([WgA, WgB], 0); Wu = np.concatenate([WuA, WuB], 0)
Wd = np.concatenate([alpha * WdA, (1 - alpha) * WdB], 1)
print("ffn merge max error      :", np.abs(ffn(xx, Wg, Wu, Wd) - tgt).max())


attention merge max error: 2.6645352591003757e-15
ffn merge max error      : 7.105427357601002e-15


## 6 · Seed → shard branching (the asynchrony enabler)

The merge *operator* works from any weights (the exactness test used pure random models). What the operator cannot guarantee by itself is merge *quality at depth*: from layer 2 on, units read a blended residual stream, and how gracefully shard-B's units interpret a stream partly written by shard-A depends on the two shards sharing a residual-stream coordinate system. Networks invent that coordinate system in the first phase of training, sensitive to data order. So we **branch every shard from one briefly-trained seed** to anchor a common basis (report §6.2).

`init_from_seed` handles the interesting case: a shard **wider** than the seed (e.g. T5's 12 heads / 3072 d_ff vs the seed's 6 / 1536). The seed's units fill the first slots; the **extra** units start *nearly silent* — their output-side weights (`wo` columns, `w_down` columns) are drawn tiny (std 1e-4) so the wide shard computes ~the seed's function at branch time, yet the new units still receive gradients and "come alive" during training. (Exact zeros would be perfectly function-preserving but would starve the new units' input weights of gradient — the classic Net2Net dead-unit problem.)


In [20]:
@torch.no_grad()
def init_from_seed(shard_cfg, seed_model):
    """Report §6.2 -- branch a shard from the common seed. If the shard is WIDER
    than the seed, the seed's units fill the first slots and the extra units start
    nearly silent (output-side weights std=1e-4): the wide shard computes ~the
    seed's function at branch time, but the new units still receive gradients."""
    m = GPT(shard_cfg)
    Hs, Fs = seed_model.cfg["n_heads"], seed_model.cfg["d_ff"]
    Hm, Fm = shard_cfg["n_heads"], shard_cfg["d_ff"]
    assert Hm >= Hs and Fm >= Fs
    m.embed.weight.copy_(seed_model.embed.weight)
    m.final_norm.weight.copy_(seed_model.final_norm.weight)
    for tb, sb in zip(m.blocks, seed_model.blocks):
        tb.attn_norm.weight.copy_(sb.attn_norm.weight)
        tb.ffn_norm.weight.copy_(sb.ffn_norm.weight)
        r = Hs * D_HEAD
        tb.attn.wq.weight[:r].copy_(sb.attn.wq.weight)
        tb.attn.wk.weight[:r].copy_(sb.attn.wk.weight)
        tb.attn.wv.weight[:r].copy_(sb.attn.wv.weight)
        tb.attn.wo.weight[:, :r].copy_(sb.attn.wo.weight)
        if Hm > Hs:
            nn.init.normal_(tb.attn.wo.weight[:, r:], std=1e-4)   # extra heads: near-silent output
        tb.ffn.w_gate.weight[:Fs].copy_(sb.ffn.w_gate.weight)
        tb.ffn.w_up.weight[:Fs].copy_(sb.ffn.w_up.weight)
        tb.ffn.w_down.weight[:, :Fs].copy_(sb.ffn.w_down.weight)
        if Fm > Fs:
            nn.init.normal_(tb.ffn.w_down.weight[:, Fs:], std=1e-4) # extra neurons: near-silent output
    return m


In [21]:
torch.manual_seed(4)
seed = GPT(SHARD_20M)                       # the narrow seed
wide = init_from_seed(SHARD_40M, seed)      # branch a WIDER family member from it
idx = torch.randint(0, VOCAB, (1, 16))
with torch.no_grad():
    diff = (wide(idx) - seed(idx)).abs().max().item()
    scale = seed(idx).std().item()
print("wide branch cfg = %s   (%.1fM params)" % (wide.cfg, wide.n_params()/1e6))
print("max |logits_wide - logits_seed| = %.4e   (logits std ~ %.3f)" % (diff, scale))
print("=> the wider shard is ~function-preserving at branch time; extra units train up later")
del seed, wide


wide branch cfg = {'n_heads': 12, 'd_ff': 3072}   (40.9M params)
max |logits_wide - logits_seed| = 8.1818e-02   (logits std ~ 0.390)
=> the wider shard is ~function-preserving at branch time; extra units train up later


## 7 · Mixing coefficients α and continual bookkeeping

αᵢ is model *i*'s voice in the function average; convexity (αᵢ ≥ 0, Σαᵢ = 1) keeps the merged output magnitude in the trained regime. The default policy is **token-proportional**: αᵢ = tokensᵢ / Σ tokens (report §7.1).

The subtle, important rule is for **continual** merging (report §7.3): when a merged model `M` (already backed by `n_M` tokens from *k* shards) absorbs a new shard `S` (`n_S` tokens), weight by **accumulated tokens** — `α_M = n_M/(n_M+n_S)`, **not** ½/½ — otherwise one newcomer drowns out *k* shards' knowledge. With this rule, any merge order yields identical final weights. The demo confirms that a 4-way simultaneous merge equals "merge A,B,C then fold in D", weight for weight.


In [22]:
# token-proportional alphas (report §7.1)
tokens = [120e6, 90e6, 150e6]
alphas = [t / sum(tokens) for t in tokens]
print("tokens", ["%.0fM" % (t/1e6) for t in tokens], "-> alphas", ["%.3f" % a for a in alphas])

# continual == simultaneous when alpha tracks accumulated tokens (report §7.3)
torch.manual_seed(3)
mods = [GPT(dict(n_heads=2, d_ff=64)) for _ in range(4)]
w = [120., 90., 150., 80.]                                  # token counts behind A, B, C, D
W = sum(w)
Msim = merge_models(mods, [wi / W for wi in w])             # one-shot 4-way
Mabc = merge_models(mods[:3], [wi / sum(w[:3]) for wi in w[:3]])
Mcon = merge_models([Mabc, mods[3]], [sum(w[:3]) / W, w[3] / W])   # fold D into the merged ABC
diff = max((p - q).abs().max().item()
           for p, q in zip(Msim.state_dict().values(), Mcon.state_dict().values()))
print("4-way simultaneous vs continual (ABC, then +D):  max weight diff = %.2e" % diff)
print("=> identical weights for any merge order under token bookkeeping")
del mods, Msim, Mabc, Mcon


tokens ['120M', '90M', '150M'] -> alphas ['0.333', '0.250', '0.417']


4-way simultaneous vs continual (ABC, then +D):  max weight diff = 7.45e-09
=> identical weights for any merge order under token bookkeeping


## 8 · End-to-end on synthetic data — the merge on *trained* weights

Everything above used random or hand-picked weights. This capstone trains **two family members independently** (no communication) on two different synthetic "partitions", then merges them — the whole paradigm in miniature, runnable in seconds on CPU.

**The toy task** is a learnable shift: most of the time `tokenₜ₊₁ = (tokenₜ + shift) mod VOCAB`, with 10% random resets. Partition A uses `shift = 1`, partition B uses `shift = 1000`. A model that learns its partition's shift drives loss far below the random-guess floor `ln(VOCAB) ≈ 9.01`. Both shards branch from the **same initial weights** (shared init), the asynchrony-friendly stand-in for a shared seed.

> **This is a mechanics demo, not a quality benchmark.** It shows (1) the merge identity holds on genuinely *trained* weights and (2) the merged model — which physically contains *both* shards' heads and neurons — handles a **mixed** stream that neither parent can. Expect the merged model to trail each specialist *slightly* on that specialist's own shift (the composition gap, §0.5) while winning decisively on the mixed stream — the SAP-Pure signature. Real perplexity on real text requires the full TinyStories run in `SAP_Test.ipynb`; SAP-Pure is deliberately unhealed here.


In [23]:
def make_partition(shift, n=24000, seed=0, p_reset=0.10):
    g = np.random.default_rng(seed)
    t = np.empty(n, dtype=np.int64); t[0] = g.integers(0, VOCAB)
    for i in range(1, n):
        t[i] = g.integers(0, VOCAB) if g.random() < p_reset else (t[i-1] + shift) % VOCAB
    return t

def get_batch(data, bs, seq):
    ix = np.random.randint(0, len(data) - seq - 1, size=bs)
    x = torch.from_numpy(np.stack([data[i:i+seq]     for i in ix]))
    y = torch.from_numpy(np.stack([data[i+1:i+1+seq] for i in ix]))
    return x.to(DEVICE), y.to(DEVICE)

def quick_train(model, data, steps=60, bs=16, seq=64, lr=3e-3, label="m"):
    model.to(DEVICE).train()
    opt = torch.optim.AdamW(model.parameters(), lr=lr, betas=(0.9, 0.95), weight_decay=0.1)
    for s in range(steps):
        x, y = get_batch(data, bs, seq)
        _, loss = model(x, y)
        opt.zero_grad(set_to_none=True); loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0); opt.step()
        if s % 20 == 0 or s == steps - 1:
            print("  [%s] step %3d  loss %.3f" % (label, s, loss.item()))
    return model

@torch.no_grad()
def eval_loss(model, data, iters=20, bs=16, seq=64):
    model.eval(); tot = 0.0
    for _ in range(iters):
        x, y = get_batch(data, bs, seq); _, loss = model(x, y); tot += loss.item()
    model.train(); return tot / iters

print("random-guess floor = ln(VOCAB) = %.3f" % math.log(VOCAB))


random-guess floor = ln(VOCAB) = 9.011


In [24]:
SHIFT_A, SHIFT_B = 1, 1000
dataA, dataB = make_partition(SHIFT_A, seed=10), make_partition(SHIFT_B, seed=20)

seed_model = GPT(SHARD_20M)                          # shared initialisation (the "seed")
print("training two shards independently (no communication):")
Ta = quick_train(copy.deepcopy(seed_model), dataA, label="T_a shift=%d" % SHIFT_A)
Tb = quick_train(copy.deepcopy(seed_model), dataB, label="T_b shift=%d" % SHIFT_B)

M = merge_models([Ta, Tb], [0.5, 0.5])              # equal tokens -> equal alpha
print("\nmerged cfg = %s  (%.1fM params)" % (M.cfg, M.n_params()/1e6))
real_merge_block_check([Ta, Tb], [0.5, 0.5], M)     # identity holds on TRAINED weights

valA   = make_partition(SHIFT_A, n=6000, seed=11)
valB   = make_partition(SHIFT_B, n=6000, seed=21)
valMix = np.concatenate([make_partition(SHIFT_A, n=3000, seed=12),
                         make_partition(SHIFT_B, n=3000, seed=22)])
print("\n  model     val(shiftA)  val(shiftB)  val(MIXED)")
for nm, mm in [("T_a", Ta), ("T_b", Tb), ("merged", M)]:
    print("  %-7s    %8.3f     %8.3f     %8.3f"
          % (nm, eval_loss(mm, valA), eval_loss(mm, valB), eval_loss(mm, valMix)))
print("\nEach parent masters only its own shift; the merged model contains BOTH sets of\n"
      "units, so on the MIXED stream it beats either parent -- knowledge was combined\n"
      "by tensor surgery alone (no gate, no router, no post-merge training).")


training two shards independently (no communication):


  [T_a shift=1] step   0  loss 9.087


  [T_a shift=1] step  20  loss 8.623


  [T_a shift=1] step  40  loss 7.784


  [T_a shift=1] step  59  loss 6.962


...

  [T_b shift=1000] step  59  loss 6.943



merged cfg = {'n_heads': 12, 'd_ff': 3072}  (40.9M params)
[real-merge check] worst sublayer error on trained weights = 3.461e-08
  (expected ~1e-9..1e-6 = float32 weight-storage rounding;
   a WRONG merge shows errors around 1e0)

  model     val(shiftA)  val(shiftB)  val(MIXED)


  T_a           7.672       11.173        9.658


  T_b          11.263        7.497        9.330


  merged        8.251        8.492        8.417

Each parent masters only its own shift; the merged model contains BOTH sets of
units, so on the MIXED stream it beats either parent -- knowledge was combined
by tensor surgery alone (no gate, no router, no post-merge training).


## 9 · Summary

### The algorithm, in one breath
Absorb each model's RMSNorm gains into its own projections (exact); **concatenate** Q/K/V columns and gate/up rows **unscaled** (input side, each unit keeps what it computes); **concatenate** `W_O` columns and `W_down` columns **scaled by αᵢ** (output side, contributions reweighted); **average** the tied embedding/final-norm (the lone parameter-level step). The result is a wider dense transformer of the same family whose every sublayer is the exact α-weighted average of its parents' sublayers.

### How this notebook satisfies the report's requirements

| # | Requirement | Realised by |
|---|---|---|
| R1 | FFN layers must merge | `merge_models` gate/up row-concat + α-scaled `w_down` (§3.2) |
| R2 | Attention layers must merge | `merge_models` Q/K/V col-concat + α-scaled `w_o` (§3.2) |
| R3 | N models merge at once | N-way `torch.cat` lists; verified 3-way & 4-way (§4.1, §5.3, §7) |
| R4 | Continual merging (merge a merge) | closure: `merge(merge(A,B),C) == merge(A,B,C)` (§5.3, §7) |
| R5 | Growth must be characterised | `S = E + NORM + Σ Pᵢ` (= the report's `E + Σ(Sᵢ−E)`), checked to the parameter (§5.2, §5.4) |
| R6 | Different-size models merge | width-heterogeneous demo, 6+3 heads (§5.2); wide branch (§6) |
| R7 | No gating networks | implicit gating via the trained nonlinearity (§5.1) |
| R8 | Real-life implementable | the operator is ~30 lines of tensor ops, training-free (§3.2) |

### The honest limits (state these in the thesis)
1. **Composition gap** — per-sublayer exactness does not compose across depth; from layer 2 on, units read a blended stream. Mitigated by the seed (§6), closed by the optional healing pass (companion notebook). This is the central testable hypothesis (report §12).
2. **Embedding average** — the single parameter-level component; kept mild by the shared seed (slow embedding drift).
3. **Final-norm gain** — cannot be absorbed into a shared averaged head; averaged instead (negligible; healing absorbs any residue).

### Deliberately *not* in this notebook (see `SAP_Test.ipynb` / the report)
The full asynchronous training pipeline (Drive, TinyStories, 30-min GPU shard runs, multi-account), the held-out perplexity comparison against the **weight-averaging baseline**, SAP-Zip (correlation-based size control, §10), SAP-Heal (the healing pass, §6.5), and GQA. This notebook is the *operator and its proof*, kept pure.

### Selected references
- Li et al., 2022. **Branch-Train-Merge.** arXiv:2208.03306 — async expert training, but ensembles (never one dense model).
- Sukhbaatar et al., 2024. **Branch-Train-MiX.** arXiv:2403.07816 — closest prior work; FFNs → MoE experts + **averaged attention** + a router that **must** be finetuned. SAP merges attention *structurally*, needs no router, and is closed under its own operator.
- Stoica et al., 2024. **ZipIt!** arXiv:2305.03053 — correlation "zipping"; reused by SAP only as optional size control, not as the merge.
- Ainsworth et al., 2023. **Git Re-Basin.** arXiv:2209.04836 — permutation alignment; made moot here because stacking never sums two units.
- Ilharco et al., 2023 (**Task Arithmetic**, arXiv:2212.04089); Yadav et al., 2023 (**TIES-Merging**, arXiv:2306.01708) — parameter-level baselines.
- Zhang et al., 2022. **Mixture of Attention Heads.** arXiv:2210.05144 — evidence that head pools work.
- Chen et al., 2016. **Net2Net.** arXiv:1511.05641 — the function-preserving widening behind `init_from_seed`.
- *Merging of Neural Networks* (Neural Processing Letters, 2024) — channel-selection merging into a same-size net; the gated contrast SAP removes by dropping the size constraint.
- *Model Merging in Pre-training of Large Language Models* (arXiv:2505.12082, 2025) — concurrent evidence that merging during pretraining is a live research direction.

*Companion: the SAP Framework Report (full derivations, §1–§16) and `SAP_Test.ipynb` (the end-to-end experiment).*
